# Связи между таблицами в СУБД на примере SQLAlchemy

## Зачем нужны отношения и внешние ключи

Мы храним разные данные в нашем приложении: пользователь, публикация, комментарий, лайк. Это всё отдельные "сущности". Некоторые сущности мы добавляем чаще, некоторые реже. Под каждую сущность мы заранее описываем структуру со строгими ограничениями по типам и значениям данных, например:

- целое число больше нуля;
- уникальная строка, допустимо отсутствие значения (NULL);
- дата и время с таймзоной;
- и так далее.

Это всё "естественные" поля, они напрямую описывают свойства сущности. А как правильно добавить связь с другой сущностью? Например:

- профиль пользователя: дополнительные поля (ваша любимая книга, цитата, и т. д.) - отдельная таблица, которая должна быть привязана к конкретному пользователю;
- автор статьи - пользователь, который написал конкретную статью;
- комментарий написан именно этим пользователем к конкретному посту;
- и так далее.

Вот чтобы добиться этой конкретики, необходимо указывать внешние ключи на соседние таблицы. Мы не хотим допускать дублирования данных. В реляционных таблицах всё построено на отношениях.

Представьте альтернативный вариант: при создании публикации мы прописываем информацию об авторе в отдельное поле в той же таблице публикаций. В таком случае на сотню постов у нас будет сотня описаний пользователя с одними и теми же значениями во всех полях: имя, почта и все остальные поля. А при изменении любой информации о пользователе нам придется обновлять все публикации, чтобы актуализировать данные. Это не только сложно, но ещё и долго и дорого.

Поэтому в реляционных СУБД принято строить строгие связи: один раз мы создали пользователя в соответствующей таблице, и каждая статья лишь ссылается на пользователя. Для этого в таблице статей достаточно хранить первичный ключ пользователя. Первичный ключ уникален и никогда не меняется в рамках жизни сущности, поэтому можно быть уверенными, что при любых изменениях данных пользователя мы сможем вытащить актуальную информацию.

А чтобы случайно не удалить сущность, на которую мы ссылаемся, или чтобы случайно не сослаться на запись, которой не существует, в реляционных СУБД есть специальные механизмы защиты - это всё про работу с внешними ключами (foreign keys). Мы можем делать это правило настолько строгим, насколько захотим.

## Связь один-ко-многим

Примеры связи один-ко-многим:

- один пользователь и много постов;
- один пост и много комментариев;
- один курс и много уроков в курсе.

Во всех этих примерах одна сущность слева может быть связана с множеством сущностей справа - от нуля до бесконечности, например:

- пользователь, у которого ещё нет постов;
- пост, у которого тысячи комментариев.

При этом не может существовать поста без автора, или комментария к посту без самого поста.

Когда мы доходим до проектирования таблицы, возникает вопрос: а где хранить связь? Кто должен отвечать за неё? И тут однозначный ответ, к которому можно прийти простым размышлением: если мы попытаемся хранить все айди постов в таблице пользователя, то нам придется работать со сложной вложенной структурой (например, с массивом). Так ещё и нужно будет обновлять таблицу пользователей каждый раз при добавлении поста. А если хранить ссылку на пользователя в таблице постов, то на каждый пост будет одно простое поле с ссылкой на автора поста - на пользователя.
Пользователь выступает родительской сущностью, а пост — зависимой. Именно поэтому ссылку на пользователя храним в модели поста.

### Проверка целостности

Если мы заведем простое числовое поле, которое должно "ссылаться" на первичный ключ в другой таблице, СУБД не станет автоматически проверять целостность. Чтобы гарантировать наличие сущности, на которую мы ссылаемся, необходимо явно прописать это в свойствах таблицы. В чистом SQL это делается с помощью ключевых слов `FOREIGN KEY` и `REFERENCES`, а в SQLAlchemy для этого есть поле `ForeignKey`. Так поле становится не просто числовым атрибутом, а ссылкой на первичный ключ пользователя. В таком случае значение этого поля ограничено записями в таблице, на которую мы ссылаемся.

Внешний ключ выполняет сразу две функции:

- выражает зависимость одной сущности от другой;
- задаёт проверяемое ограничение на уровне базы данных.

### Внешний ключ на SQLAlchemy ORM модели

Для работы с внешним ключом воспользуйтесь специальным типом `ForeignKey`. В него передайте колонку, на которую нужно ссылаться. Это может быть обращение напрямую через связанную модель, но зачастую это строка формата `[схема].таблица.колонка`, где схема опциональна, а имя таблицы и колонки должны обязательно присутствовать.

Пример:

```python
class Post(Base):  
    ...

    user_id: Mapped[int] = mapped_column(  
        ForeignKey("user.id"),  
    )
```

Так запись без настоящего `user_id` будет недопустимой - такой пост нельзя будет сохранить в таблице.



> [!WARNING] Про внешние ключи в SQLIte
>
> Учитывайте особенности SQLite: почти все фишки там надо включать вручную в рамках сессии. Это сделано для обратной совместимости и явности. Если вам нужны проверки при работе с внешними ключами (а они вам очень нужны), необходимо включить их поддержку через `PRAGMA foreign_keys=ON`. Причём не один раз вручную, а автоматически при каждом новом подключении.
>
> В рамках SQLAlchemy необходимо делать это на уровне `engine`.
> Тогда любая попытка сохранить пост с несуществующим `user_id` будет приводить к ошибке.

  

### Доступ к связанной сущности через `relationship`

Одно из удобств SQLAlchemy ORM это объектное представление связи: мы можем получать доступ к связанной сущности через понятное питонячье свойство.

Добавьте свойство `relationship` на ORM модель. Это поле не добавляет новые столбцы. Оно нужно только чтобы описать существующую связь для ORM и сделать связанные объекты доступными в виде обычных свойств.
На стороне поста такая связь даёт доступ к автору через `post.user`, а на стороне пользователя к коллекции постов через `user.posts`.

Тем самым отношение между таблицами преобразуется в отношение между объектами, что делает работу с моделью ближе к логике Python-кода и отдаляет от более сложного ручного управления идентификаторами (айдишниками сущностей).


Чтобы связь работала согласованно в обе стороны, необходимо определить `back_populates`. На примере связи пользователь-пост
то свойство связывает атрибут пользователя, содержащий посты, и атрибут поста, содержащий пользователя, как две стороны одного и того же отношения. Это важно потому, что без такого согласования ORM воспринимала бы их как независимые свойства. Благодаря `back_populates` мы получаем полноценную двустороннюю навигацию на модели: от поста можно перейти к автору, а от пользователя к его постам.
Корректно определить значения очень просто: на левой модели в `back_populates` нужно указать имя свойства, как с правой модели нужно будет получать доступ к этой левой модели, и наоборот. Поэтому для `user` в связи с постами в поле `back_populates` будет значение `"user"`, а для поста в связи с пользователем в поле `back_populates` будет значение `"posts"`:

```python
# user.py
class User(Base):  
    ...  

    posts: Mapped[list[Post]] = relationship(  
        back_populates="user",  
    )

# post.py
class Post(Base):
    ...

    user_id: Mapped[int] = mapped_column(
        ForeignKey("user.id"),
    )

    user: Mapped[User] = relationship(
        back_populates="posts",
    )
```

Читаем это так:

- с пользователя доступ к постам через поле `posts` И именно `"posts"` будет в `back_populates` на посте в связи к пользователю. Чтобы с поста вернуться к пользователю (автору), нужно обратиться к `post.user`, поэтому в `back_populates` указываем значение `"user"`;
- с поста доступ к пользователю через поле `user` - как в `back_populates` в связи к постам на пользователе. Чтобы с пользователя перейти к его постам, нужно обратиться к `user.posts`, поэтому в `back_populates` указываем значение `"posts"`.


### Сессия SQLAlchemy
  
При связывании поста с объектом пользователя через ORM оба объекта должны находиться в одной сессии. ORM отслеживает состояние сущностей в пределах общего контекста работы.
Если пользователь получен в одной сессии, а пост создаётся в другой, связать их не получится, надо будет сначала привязать обе сущности к одной сессии.


### Загрузка связанных записей и проблема скрытых SQL-запросов

Вот мы объявили связь и хотим вычитать данные, включая связанные сущности. Например, загружаем посты и хотим отобразить посты вместе с авторами.
Обращение к связанному атрибуту выполнит неявный вызов SQL-запроса. Если сначала выполнить `select(Post)`, а затем для каждого объекта поста обращаться к `post.user`, ORM будет подгружать пользователей отдельными запросами.

Это удобно, потому что приложение продолжит работать, но это крайне неоптимально, и обязательно приведет к проблемам с производительностью. Поэтому для работы со связями необходимо заранее определить, к каким связанным сущностям мы будем обращаться, и загрузить их заранее.
  

> [!NOTE] Проблема N+1
> Когда после одного запроса за основным набором сущностей выполняется ещё по одному запросу для каждой связанной записи, возникает проблема N+1. В модели постов и пользователей это проявляется тогда, когда автор каждого поста загружается отдельно.
>
> Формула проста: одним запросом загружаем N постов (например, десять штук), а потом на каждый из постов шлём SQL запрос, чтобы вытащить информацию об авторе. Получается, нужно сделать ещё N запросов, чтобы получить всю необходимую информацию. Выходит 1 + N запросов. Это и называется проблемой N+1, потому что для N=10 у нас будет 11 запросов, а для N=100 у нас будет 101 запрос.
>
> **Дорого выполнять много запросов там, где можно обойтись одним или двумя. Этого надо избегать.**


### Подгрузка единичной связанной сущности

  
Когда требуется получить посты вместе с авторами, используйте опцию подгрузки `joinedload`. В этом случае связь к пользователю загружается сразу в составе основного запроса. Для отношения «многие посты — один пользователь» такой способ естественен: у каждого поста автор ровно один.

```python
from sqlalchemy import select
from sqlalchemy.orm import joinedload

stmt = (  
    select(Post)  
    .options(  
        joinedload(Post.user),  
    )  
    .order_by(  
        Post.title,  
        Post.id.asc(),  
    )  
)
```

Теперь обращение к `post.user` не выполняет дополнительные запросы. Таким образом, `joinedload` решает задачу предсказуемой и компактной загрузки единичной связанной сущности.

#### Почему `joinedload` подходит для связи к одному объекту


При единичной связи строка основного результата может быть расширена данными связанного объекта без усложнения структуры выборки. Повторение данных автора возможно, если один и тот же пользователь написал несколько постов, однако в такой конфигурации это менее затратно, чем выполнение отдельного запроса на каждого автора. Поэтому для направления «от одного поста **к одному** пользователю» `joinedload` оказывается подходит лучше всего. Здесь стратегия загрузки следует за кардинальностью связи: к одному связанному объекту удобнее присоединяться сразу, чем подгружать его отдельно.

> [!NOTE] Когда выбирать `joinedload`
> Видим связь "к одному" - выбираем `joinedload`.

### Загрузка коллекции связанных записей и проблема скрытых SQL-запросов

Читаем пользователей вместе с их постами. Теперь на стороне связи находится коллекция из постов.

Если идти по всем полученным пользователям и обращаться к `user.posts` то, SQLAlchemy ORM будет делать отдельный запрос постов для каждого пользователя. Даже в тех случаях, когда коллекция пуста, так как мы заранее не знаем, есть ли у пользователя посты.

Если загружать связанные посты через `joinedload`, прямое соединение приведёт к повторению пользователя по числу его постов.
Поэтому применяем другой способ: сначала запрашиваем пользователей, а затем одним отдельным запросом загружаем все посты для всех вытащенных пользователей. Эти шаги автоматически выполняются при использовании `selectinload`. Этот тип подгрузки лучше всего использовать там, где к одной сущности привязано произвольное число объектов: от нуля до бесконечности.
Для связи «один пользователь — много постов» это стандартный способ контролируемой загрузки коллекции.

```python
from sqlalchemy import select
from sqlalchemy.orm import selectinload

stmt = (
    select(User)
    .options(
        selectinload(User.posts),
    )
    .order_by(
        User.username,
    )
)
```


> [!NOTE] Когда выбирать `selectinload`
> Видим связь "ко многим" - выбираем `selectinload`.

### Фильтрация по связанной таблице

Связь между таблицами можно использовать для фильтрации по свойствам связанных сущностей. Например, можно отбирать пользователей по условиям, наложенным на посты: "дай всех пользователей, кто писал посты про Python". В таком случае связь становится частью логики запроса.
Здесь особенно важно различать два уровня: какие сущности выбираются в основной результат и какие связанные данные затем будут подгружены.

```python
stmt = (
    select(User)
    .join(
        User.posts,
    )
    .where(
        Post.title.like("%Python%"),
    )
    .options(
        selectinload(User.posts),
    )
    .order_by(
        User.username,
    )
)
```
  

Пользователи будут отобраны по признаку, что у каждого есть пост с "Python" в названии. Но в `user.posts` будут все посты пользователя.
Условие, по которому сущность попала в основной набор, и состав уже подгруженной коллекции — разные аспекты запроса. Фильтрация по связанной таблице не приводит к фильтрации содержимого ORM-связи.
  

## Запомнить

- В модели «пользователь — посты» зависимость закрепляется на уровне таблицы через `ForeignKey`, а на уровне ORM — через `relationship` и `back_populates`.
- В SQLite объявление внешнего ключа недостаточно само по себе: проверка должна включаться через `PRAGMA foreign_keys=ON` при каждом подключении.
- Доступ к связанным объектам должен проектироваться заранее, иначе ORM порождает скрытые дополнительные запросы: проблему N+1.
- Для подгрузки "к одному" лучше подходит `joinedload`, а для подгрузки "ко многим" выбирайте `selectinload`.
